# マスク言語モデルで誤りのありそうな箇所を見つける（文法誤り検出）

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/27_gec_education/ged_mlm.ipynb)

NLP クックブック 27-10 ／ 文法誤り訂正と言語教育

訂正まではしなくてよいので、文のどこに誤りがありそうかを指摘したい。誤りの注釈付きデータを使わずに、学習済みの言語モデルだけで行う。

In [1]:
!pip install -q fugashi unidic-lite datasets

In [2]:
import torch
from transformers import AutoModelForMaskedLM, AutoTokenizer

name = "tohoku-nlp/bert-base-japanese-v3"
tokenizer = AutoTokenizer.from_pretrained(name)
model = AutoModelForMaskedLM.from_pretrained(name).eval()

@torch.no_grad()
def token_scores(sentence):
    """各トークンを 1 つずつ [MASK] に置き換え、元のトークンが予測される確率と、最も有力な候補を調べる"""
    ids = tokenizer(sentence, return_tensors="pt")["input_ids"][0]
    positions = range(1, len(ids) - 1)
    batch = ids.repeat(len(positions), 1)
    for row, pos in enumerate(positions):
        batch[row, pos] = tokenizer.mask_token_id
    probs = model(input_ids=batch).logits.softmax(-1)
    out = []
    for row, pos in enumerate(positions):
        p = probs[row, pos]
        out.append((tokenizer.convert_ids_to_tokens(int(ids[pos])), float(p[ids[pos]]), tokenizer.convert_ids_to_tokens(int(p.argmax()))))
    return out

def detect(sentence, threshold=0.01):
    marked = []
    for token, p, best in token_scores(sentence):
        token = token.replace("##", "")
        marked.append(f"【{token}→{best.replace('##', '')}?】" if p < threshold else token)
    return "".join(marked)

for s in ["図書館で本が借りました。", "私は毎日学校を行きます。", "昨日は友達と映画を見ます。", "この料理はとてもおいしいでした。",
          "彼は日本語が上手に話せます。", "私は東京に住んでいます。", "明日は雨が降るでしょう。"]:
    print(detect(s))

図書館で【本→私?】【が→を?】【借り→見つかり?】ました。
私は【毎日→この?】学校【を→に?】【行き→休み?】ます。
昨日は友達と映画を見【ます→た?】。
この料理はとてもおいしいでした。
彼は日本語が上手に話せます。
私は東京に住んでいます。
明日は雨が降るでしょう。


In [3]:
# トークンごとの確率を見る
for token, p, best in token_scores("図書館で本が借りました。"):
    print(f"{token}\t{p:.4f}\t最有力: {best}")

図書	0.9441	最有力: 図書
館	0.8028	最有力: 館
で	0.6340	最有力: で
本	0.0067	最有力: 私
が	0.0006	最有力: を
借り	0.0009	最有力: 見つかり
まし	0.0215	最有力: られ
た	0.9925	最有力: た
。	0.9959	最有力: 。


In [4]:
# 誤りを含む文と訂正済みの文で、検出される箇所の数を比べる (検出の当たり外れの目安)
from datasets import load_dataset

ds = load_dataset("ronantakizawa/jfleg-japanese", split="validation").select(range(150))

def n_flagged(sentence, threshold):
    return sum(p < threshold for _, p, _ in token_scores(sentence))

for threshold in [0.001, 0.01, 0.05]:
    wrong = [n_flagged(ex["sentence"], threshold) > 0 for ex in ds]
    right = [n_flagged(ex["corrections"][0], threshold) > 0 for ex in ds]
    tp, fp = sum(wrong), sum(right)
    print(f"しきい値 {threshold}: 誤りのある文で指摘あり {tp / len(ds):.2f} / 正しい文で指摘あり (誤検出) {fp / len(ds):.2f} / 適合率 {tp / (tp + fp):.2f}")

しきい値 0.001: 誤りのある文で指摘あり 0.51 / 正しい文で指摘あり (誤検出) 0.16 / 適合率 0.76
しきい値 0.01: 誤りのある文で指摘あり 0.83 / 正しい文で指摘あり (誤検出) 0.57 / 適合率 0.59
しきい値 0.05: 誤りのある文で指摘あり 0.97 / 正しい文で指摘あり (誤検出) 0.81 / 適合率 0.54


- マスク言語モデルは「この位置にこの語が来る確率」を出せる。確率が極端に低い語は、文脈に合っていない、つまり誤りの可能性がある。誤りの注釈データなしで使える。
- 助詞の誤り（「本が借りました」）のように、周りの語から正解がほぼ決まる誤りは見つけやすい。内容語の選び間違いや、語の抜け・余分は見つけにくい。
- 珍しい固有名詞や専門用語も確率が低くなるので、誤りと区別できない。しきい値を下げると見逃しが増え、上げると誤検出が増える。
- 誤りの位置を正確に指すとは限らない（誤った助詞ではなく、その後ろの動詞の確率が下がることがある）。「このあたりが怪しい」という手がかりとして使う。
- 高い精度が必要なら、誤りの注釈付きデータで系列ラベリング（各トークンが正しいか誤りか）を学習する。固有表現抽出の章の BERT のファインチューニングと同じ方法が使える。